# Case Studies

In [ ]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.config import *
from src.load_save_data import *
from src.preprocess_data import *
from src.explore_data import *
from src.feature_engineering import *
from src.performance_analysis import *

# Case Study 1: Large Performance Gap

Deconstructing a 0.93-second gap at the 2025 Japanese Grand Prix (Q1)

During Q1 of the 2025 Japanese Grand Prix, a massive 0.93-second gap separated Fernando Alonso (1:28.337) and Lance Stroll (1:29.271). This section breaks down exactly where and how that time was extracted.

In [ ]:
TEAM = "Aston Martin"
GRAND_PRIX = "Japanese Grand Prix"
YEAR = 2025

In [ ]:
session = fastf1.get_session(YEAR, GRAND_PRIX, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, TEAM)
segment = get_segment(session, driver_1, driver_2)
turns = get_turns(session)

try:
    lap_1, lap_2 = load_cache(YEAR, GRAND_PRIX, segment, driver_1, driver_2)
except Exception as e:
    print(f"Processed data for {YEAR} {GRAND_PRIX} {driver_1} and {driver_2} not found: {e}")

## Laps' Information

In [ ]:
print(f"=== Teammate Analysis for {YEAR} {GRAND_PRIX} ===")
print(f"Driver 1: {driver_1}, Lap Time: {lap_1['LapTime']}, Tyres: {lap_1['TyreCompound']}, Tyre Age: {lap_1['TyreAge']}")
print(f"Driver 2: {driver_2}, Lap Time: {lap_2['LapTime']}, Tyres: {lap_2['TyreCompound']}, Tyre Age: {lap_2['TyreAge']}")

print(f"Time delta between {driver_1} and {driver_2}: {-(lap_1['LapTime'] - lap_2['LapTime']):.3f} seconds")

Despite both drivers running the exact same tyre compound and identical tyre age, a massive 0.93-second gap still remains!

## Time Delta Analysis 

In [ ]:
plot_variable_delta(lap_1, lap_2, "Time", turns, y_unit="s")

compare_times(lap_1, lap_2)

barplot_feature_comparison(lap_1, lap_2, "SegmentTime", y_unit="s")
barplot_feature_delta(lap_1, lap_2, "SegmentTime", y_unit="s")
barplot_feature_comparison(lap_1, lap_2, "TurnTime", y_unit="s")
barplot_feature_delta(lap_1, lap_2, "TurnTime", y_unit="s")

In the first sector, Alonso demonstrates consistent and cumulative time gains. Because these corners naturally flow into one another, carrying momentum through this sequence is critical. Alonso manages to string these linked corners together, compounding his speed advantage to build an early lead. Moving further into the lap, Turn 13 stands out as a distinct braking event where he secures an isolated gain of 0.112 seconds.

The undisputed highlight of this analysis is the zone spanning Turns 14 through 16, where the time delta fluctuates wildly. The sequence begins at Turn 14, where Alonso extracts a massive 0.263-second advantage over Stroll. However, the dynamic shifts dramatically immediately after. At Turn 15, Stroll recovers some amount of ground, causing Alonso to drop 0.187 seconds in this single micro-sector. This sharp swing is clearly visible as a steep upward spike near the end of the delta line graph. Yet, Alonso retaliates just as quickly at Turn 16, crushing the sector time once again by pulling another 0.220 seconds ahead.

This absolute rollercoaster in the time delta strongly points to drastically different driving lines and overall approaches to this complex. The initial hypothesis is that we are witnessing a deliberate strategic compromise. It appears highly likely that Alonso intentionally sacrifices his minimum speed and overall time through the 14 and 15 sequence to guarantee superior car positioning and a perfectly optimized braking phase for Turn 16. By doing so, he establishes a much stronger platform for traction, maximizing his exit speed onto the ensuing straight and turning a short-term loss into an overall gain.

## Possible Main Contributor Features

In [ ]:
priority_segments = find_priority_zones(lap_1, lap_2, feature_type="Segments")
priority_turns = find_priority_zones(lap_1, lap_2, feature_type="Turns")

contributors_segments = find_feature_contributors(lap_1, lap_2, priority_segments, feature_type="Segments",)
contributors_turns = find_feature_contributors(lap_1, lap_2, priority_turns, feature_type="Turns",)

summary_segments = calculate_contributor_summary(priority_segments, contributors_segments, feature_type="Segments",)
summary_turns = calculate_contributor_summary(priority_turns, contributors_turns, feature_type="Turns",)

display(summary_segments)
display(summary_turns)

## Turns 1 to 12

In [ ]:
compare_features(lap_1, lap_2, ["BrakingPoint", "EntrySpeed", "ApexSpeed", "ExitSpeed"])
barplot_feature_delta(lap_1, lap_2, "EntrySpeed", "m/s")
barplot_feature_delta(lap_1, lap_2, "ApexSpeed", "m/s")
barplot_feature_delta(lap_1, lap_2, "ExitSpeed", "m/s")

At Turn 1, both drivers brake at the same point, 31.9 metres before the corner. Alonso nevertheless carries more speed through the corner, with an entry speed of 316.9 km/h compared to Stroll's 310.7 km/h. He also maintains a higher apex speed, 226.0 versus 221.8 km/h, and exits faster at 218.8 versus 215.9 km/h. Despite these differences, the resulting turn times are almost identical.

The sequence through Turns 6–8 highlights differences in corner speed and braking approach. At Turn 6, Alonso brakes earlier, at 53.2 metres compared to Stroll's 44.6 metres. Although Stroll carries slightly more speed at the apex, the exit speeds are almost identical, with Alonso at 193.3 km/h and Stroll at 192.9 km/h. The difference becomes clearer at Turn 7, where Alonso carries a higher apex speed of 248.3 km/h compared to Stroll's 240.5 km/h and reaches a higher exit speed of 263.9 versus 259.0 km/h. Turn 8 also shows Alonso carrying stronger corner speed, with both drivers braking at the same 33.3-metre point.

Turn 9 provides the clearest example of a lower-entry-speed, higher-exit-speed approach. Alonso enters at 222.5 km/h compared to Stroll's 240.6 km/h and reaches a slightly lower apex speed, 131.6 versus 133.5 km/h. However, Alonso exits at 156.7 km/h, more than 12 km/h faster than Stroll's 144.2 km/h. Both drivers brake at the same 68.6-metre point, indicating that the difference is concentrated in the corner phases after braking rather than in the initial braking point.

Turn 10 shows a much smaller difference in corner execution. Neither driver has a detected braking zone, and their apex and exit speeds remain relatively close. Alonso reaches a slightly higher apex speed of 260.7 km/h compared to Stroll's 258.0 km/h, while Stroll holds a small advantage at the exit, 246.9 versus 244.3 km/h. Alonso nevertheless completes the turn 0.021 seconds faster.

Turn 11 again shows a substantial difference between entry and exit speed. Both drivers brake at approximately 99.1 metres, but Alonso enters slower at 157.6 km/h compared to Stroll's 167.9 km/h. He reaches a similar apex speed, 71.5 versus 69.5 km/h, before producing a significantly higher exit speed of 128.1 km/h compared to Stroll's 116.2 km/h. Despite this 11.9 km/h exit advantage, the overall turn times are almost identical, with Stroll actually being faster.

Overall, Turns 1–12 suggest that the largest differences are concentrated in specific phases of individual corners rather than in a consistent braking-point advantage. Alonso often combines lower entry speed with higher exit speed, most clearly at Turns 9 and 11. However, these speed differences do not always translate directly into lower turn time, highlighting the importance of analysing the complete corner rather than treating entry, apex, or exit speed independently.

## Turns 13 to 18

In [ ]:
compare_features(lap_1, lap_2, ["FullThrottlePercentage"])
barplot_feature_comparison(lap_1, lap_2, "BrakingPoint", "m")
barplot_feature_delta(lap_1, lap_2, "FullThrottlePercentage", "%")

The data through Turns 13–15 shows a mixed picture in corner approach. At Turn 13, Stroll carries substantially more entry speed, entering at 296.1 km/h compared to Alonso's 276.5 km/h. However, Alonso reaches a slightly higher exit speed of 230.4 km/h versus 227.9 km/h. Stroll also spends longer braking, 1.27 seconds compared to Alonso's 0.98 seconds, showing a clear difference in how the two drivers manage the deceleration phase.

Turn 14 follows a similar pattern, with Alonso producing a much stronger exit speed of 197.6 km/h compared to Stroll's 178.9 km/h. This is one of the larger exit-speed differences in the section, despite the relatively small difference in entry speed.

Turn 15 shows very little difference in overall performance. Stroll's turn time is 1.153 seconds compared to Alonso's 1.148 seconds, while both drivers record very similar apex and exit speeds. The difference is therefore negligible in terms of the individual corner.

Turn 16 is the clearest performance separator in this section. Alonso brakes earlier, beginning at 124.7 metres compared to Stroll's 107.5 metres, and enters at 113.2 km/h versus Stroll's 157.5 km/h. Despite sacrificing more than 44 km/h on entry, Alonso achieves a much stronger exit, reaching 128.8 km/h compared to Stroll's 101.8 km/h. The corner times, however, show that this approach does not make Alonso faster through the corner itself.

The difference in exit speed is nevertheless visible in the following section. At Turn 17, Alonso reaches 199.6 km/h compared to Stroll's 180.5 km/h, although Stroll records the faster turn time. Turn 18 is again very close.

Overall, Turns 13–18 show that Alonso's largest advantage in this section comes from producing higher exit speeds in specific corners, particularly Turns 14 and 16. However, these speed advantages do not consistently translate into faster individual turn times, making it more accurate to describe them as differences in corner execution rather than direct time gains through the corners themselves.

## Case Study Conclusion

The aggregate telemetry analysis shows that Alonso's advantage over Stroll is concentrated in specific sections of the lap rather than being produced by a consistent difference in braking points. The strongest differences appear in corner exit performance, with Alonso frequently achieving higher exit speeds even when his entry or apex speed is lower.

The braking data also shows that Alonso does not rely consistently on later braking. At Turn 6, Alonso brakes at 53.2 metres compared to Stroll's 44.6 metres, while at Turn 16 the difference is even larger, with Alonso braking at 124.7 metres compared to Stroll's 107.5 metres. This suggests that braking point selection is being used differently across the lap, rather than Alonso simply braking later to gain time.

Overall, the data points towards a performance advantage concentrated in specific corners and corner sequences. Alonso's strongest differences are visible in the transition from apex to exit, where he is able to produce substantially higher exit speeds in several key corners. The results therefore suggest that the decisive differences between the two laps come from how each driver manages the corner phases and carries speed into the following sections, rather than from a single consistent advantage in braking or entry speed.

# Case Study 2: Small Performance Gap

Deconstructing a 0.015-second gap at the 2025 British Grand Prix (Q3)

During Q3 of the 2025 British Grand Prix, a tiny 0.015-second gap separated Oscar Piastri (1:24.995) and Lando Norris (1:25.010). This section breaks down exactly where and how that time was extracted.

In [ ]:
TEAM = "McLaren"
GRAND_PRIX = "British Grand Prix"
YEAR = 2025

In [ ]:
session = fastf1.get_session(YEAR, GRAND_PRIX, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, TEAM)
segment = get_segment(session, driver_1, driver_2)
turns = get_turns(session)

try:
    lap_1, lap_2 = load_cache(YEAR, GRAND_PRIX, segment, driver_1, driver_2)
except Exception as e:
    print(f"Processed data for {YEAR} {GRAND_PRIX} {driver_1} and {driver_2} not found: {e}")

## Laps' Information

In [ ]:
print(f"=== Teammate Analysis for {YEAR} {GRAND_PRIX} ===")
print(f"Driver 1: {driver_1}, Lap Time: {lap_1['LapTime']}, Tyres: {lap_1['TyreCompound']}, Tyre Age: {lap_1['TyreAge']}")
print(f"Driver 2: {driver_2}, Lap Time: {lap_2['LapTime']}, Tyres: {lap_2['TyreCompound']}, Tyre Age: {lap_2['TyreAge']}")

print(f"Time delta between {driver_1} and {driver_2}: {-(lap_1['LapTime'] - lap_2['LapTime']):.3f} seconds")

Both drivers were running the exact same tyre compound and identical tyre age, and despite having different setups and driving styles only a tiny 0.015-second gap separates them.

In [ ]:
plot_variable_delta(lap_1, lap_2, "Time", turns, y_unit="s")

compare_times(lap_1, lap_2)

barplot_feature_comparison(lap_1, lap_2, "SegmentTime", y_unit="s")
barplot_feature_delta(lap_1, lap_2, "SegmentTime", y_unit="s")
barplot_feature_comparison(lap_1, lap_2, "TurnTime", y_unit="s")
barplot_feature_delta(lap_1, lap_2, "TurnTime", y_unit="s")

The lap begins with a relatively neutral entry through Turn 1, but Turn 2 immediately introduces a massive swing in performance. Norris extracts a massive 0.297 seconds over Piastri here, creating a significant and sharp upward spike in the delta graph. However, this advantage is almost entirely neutralized just as quickly. After a marginal 0.032-second gain for Norris in Turn 3, Piastri retaliates heavily in Turn 4, clawing back an impressive 0.284 seconds. Piastri carries this momentum slightly into Turn 5, edging out another 0.029 seconds. This extreme, high-volatility exchange suggests drastically different racing lines or a strategic compromise, showcasing how a near-identical overall lap time can be built on completely contrasting micro-sectors.

Moving further into the lap, the pendulum swings again, though with slightly less violence than the opening complex. Turn 6 sees Norris reclaim 0.112 seconds, indicating a stronger entry or better mid-corner rotation. Piastri immediately counters this in Turn 7, recovering 0.081 seconds, and manages to extend this localized advantage through Turn 8 by taking another 0.023 seconds. Unlike the massive fluctuations in the opening sequence, this sector represents a more rhythmic give-and-take. The drivers are trading smaller fractions of a second, likely optimizing different specific phases of this flowing corner sequence.

This middle section of the track is initially characterized by extreme marginality before a distinct shift in momentum. Turns 10 and 11 are practically identical, with Norris gaining a microscopic 0.003 and 0.013 seconds, respectively. Norris builds on this slightly in Turn 12, adding 0.043 seconds to his advantage. However, as the sequence transitions into the subsequent braking and traction zones, Piastri begins to shift the delta. After a negligible 0.010-second edge in Turn 13, Piastri secures a solid 0.080-second gain in Turn 14. This strongly points to superior braking stability or an earlier, cleaner throttle application that allows Piastri to carry more speed out of this specific complex.

The final sector perfectly encapsulates the fragmented micro-battle that defines this qualifying gap. Norris attacks Turn 15 effectively, securing a 0.078-second gain, before they match each other almost perfectly through Turn 16 (a neutral 0.005-second difference). The decisive late-lap counter-punch from Piastri arrives at Turn 17, where he sharply takes 0.116 seconds. This suggests Piastri highly optimized his exit trajectory leading onto the final straight, a momentum advantage he compounds with a final 0.021-second gain through Turn 18. 

## Possible Main Contributor Features

In [ ]:
priority_segments = find_priority_zones(lap_1, lap_2, feature_type="Segments")
priority_turns = find_priority_zones(lap_1, lap_2, feature_type="Turns")

contributors_segments = find_feature_contributors(lap_1, lap_2, priority_segments, feature_type="Segments",)
contributors_turns = find_feature_contributors(lap_1, lap_2, priority_turns, feature_type="Turns",)

summary_segments = calculate_contributor_summary(priority_segments, contributors_segments, feature_type="Segments",)
summary_turns = calculate_contributor_summary(priority_turns, contributors_turns, feature_type="Turns",)

display(summary_segments)
display(summary_turns)

## Turns 1 to 5

In [ ]:
compare_features(lap_1, lap_2, ["EntrySpeed", "ApexSpeed", "ExitSpeed"])
compare_features(lap_1, lap_2, ["BrakingPoint", "BrakingDistance", "BrakingDuration"])  
barplot_feature_delta(lap_1, lap_2, "EntrySpeed", "m/s")
barplot_feature_delta(lap_1, lap_2, "ApexSpeed", "m/s")
barplot_feature_delta(lap_1, lap_2, "ExitSpeed", "m/s")

Through Turns 1 and 2, both drivers show an almost identical high-speed profile. Neither driver uses the brakes, with both remaining in 8th gear and spending 100% of the segment at full throttle. Turn 1 is particularly close, with Piastri completing it in 1.202 seconds compared to Norris's 1.205 seconds. At Turn 2, both drivers reach an apex speed of 305.0 km/h, while Piastri records a slightly higher mean throttle of 98.7% compared to Norris's 98.7%, leaving virtually no meaningful difference between the two.

Turn 3 marks the first major separation in the data. Norris carries substantially more speed into the corner, entering at 291.1 km/h compared to Piastri's 246.5 km/h. However, this comes with a longer braking phase, with Norris covering 73.9 metres over 1.60 seconds compared to Piastri's 55.5 metres over 1.24 seconds. Norris also reaches a higher apex speed of 130.3 km/h against Piastri's 123.0 km/h, but Piastri completes the corner more quickly overall.

Turn 4 shows a different approach, with Norris braking at 42.7 metres compared to Piastri's 70.4 metres. Despite the later braking point, Norris records a higher minimum speed of 89.8 km/h compared to Piastri's 91.8 km/h, while Piastri achieves a substantially stronger exit at 139.4 km/h versus Norris's 117.0 km/h. Piastri also records a higher mean throttle of 84.8% compared to Norris's 79.7%, indicating a stronger acceleration phase through the corner.

Turn 5 is much closer in overall performance. Piastri enters at 199.3 km/h compared to Norris's 192.1 km/h and reaches a higher apex speed of 204.0 km/h against 200.4 km/h. The exit remains close, with Piastri reaching 245.0 km/h compared to Norris's 243.5 km/h. Piastri also records a higher mean throttle of 86.2% compared to Norris's 83.5%, although the resulting turn-time difference is minimal.


## Turns 6 to 8

In [ ]:
barplot_feature_delta(lap_1, lap_2, "BrakingPoint", "m")
barplot_feature_delta(lap_1, lap_2, "BrakingDuration", "s")
barplot_feature_delta(lap_1, lap_2, "FullThrottlePercentage", "%")

At Turn 6, Norris takes a more aggressive approach to the corner, braking later at 109.8 metres compared to Piastri's 91.3 metres. This allows Norris to enter faster at 258.7 km/h against Piastri's 254.4 km/h, but the additional entry speed comes with a substantially longer braking phase, covering 120.2 metres over 1.81 seconds compared to Piastri's 83.2 metres over 1.26 seconds. Despite the difference in entry speed, Piastri reaches a slightly higher apex speed of 181.9 km/h compared to Norris's 181.0 km/h.

The balance shifts through Turn 7. Both drivers brake at the same 107.4-metre point and cover the same 55.5-metre braking distance, but Norris enters faster at 144.8 km/h compared to Piastri's 137.8 km/h. The apex speeds are almost identical, with Norris at 120.3 km/h and Piastri at 119.1 km/h. Piastri nevertheless produces the stronger exit, reaching 163.8 km/h against Norris's 156.7 km/h. His higher throttle commitment also reflects this difference, with Piastri reaching 83.8% full throttle compared to Norris's 81.1%.

The advantage generated through Turn 7 carries directly into Turn 8. Both drivers take the corner flat-out in 8th gear, but Piastri maintains a higher speed profile, entering at 269.1 km/h compared to Norris's 267.2 km/h and exiting at 288.3 km/h against 286.1 km/h. The combination of the stronger Turn 7 exit and slightly higher speed through Turn 8 allows Piastri to recover the time lost in the preceding corner and finish the Turn 7–8 sequence faster.


## Turns 9 to 14

Through Turns 10–12, Norris holds a small but consistent advantage in high-speed cornering. In Turn 10, both drivers remain flat-out in 8th gear with 100% full throttle, but Norris carries slightly higher speed, entering at 310.7 km/h and reaching a minimum of 305.5 km/h compared to Piastri's 308.0 km/h and 301.8 km/h. Turn 11 shows a similar pattern, with Norris using more throttle throughout the segment, recording 91.2% mean throttle and 66.7% full throttle compared to Piastri's 85.3% and 53.3%. This translates into small gains for Norris.

The advantage continues into Turn 12, where both drivers use 6th gear. Norris enters at 297.0 km/h and maintains a minimum speed of 276.9 km/h, compared to Piastri's 295.8 km/h and 272.2 km/h. The higher minimum speed allows Norris to complete the segment faster, extending his advantage before the heavier braking zones.

Turn 13 marks the change in momentum. Norris enters faster at 269.3 km/h compared to Piastri's 259.6 km/h, but requires a much longer braking phase, covering 46.2 metres over 0.67 seconds. Piastri uses only a 9.2-metre braking phase lasting 0.13 seconds, allowing him to complete the deceleration earlier and reach a slightly higher exit speed of 239.2 km/h compared to Norris's 238.0 km/h. His higher throttle commitment, with 80.0% full throttle compared to Norris's 66.7%, also helps him recover time.

Piastri carries this momentum into Turn 14, where he brakes earlier at 100.3 metres compared to Norris's 81.8 metres and reaches a higher minimum speed of 249.4 km/h against 247.1 km/h. He also spends 87.2% of the segment at full throttle compared to Norris's 85.1%, producing a stronger exit of 270.1 km/h versus 266.3 km/h. This gives Piastri an advantage through Turn 14 and reverses the small deficit accumulated across Turns 10–12.


## Turns 15 to 18

In [ ]:
compare_features(lap_1, lap_2, ["MinimumSpeed", "MeanThrottle", "FullThrottlePercentage"])
barplot_feature_delta(lap_1, lap_2, "MinimumSpeed", "km/h")
barplot_feature_delta(lap_1, lap_2, "MeanThrottle", "%")
barplot_feature_delta(lap_1, lap_2, "FullThrottlePercentage", "%")

Turn 15 gives Norris another small advantage in the final sector. He enters at 285.3 km/h compared to Piastri's 283.7 km/h and maintains a higher minimum speed of 253.8 km/h against 250.9 km/h. Norris also commits more heavily to the throttle, with 76.4% mean throttle and 60.4% full throttle compared to Piastri's 74.6% and 54.2%. This produces a stronger exit of 252.6 km/h versus 250.3 km/h and gives Norris the edge.

The balance becomes almost neutral through Turn 16. Norris carries significantly more entry speed, at 286.0 km/h compared to Piastri's 264.5 km/h, by braking much later at 39.2 metres compared to 76.1 metres. Piastri therefore spends longer decelerating, covering 101.7 metres over 2.16 seconds compared to Norris's 73.9 metres over 1.78 seconds. Despite the difference in approach, Piastri reaches a similar exit speed of 112.7 km/h, marginally ahead of Norris's 112.2 km/h, leaving only a small difference between them.

Turn 17 is where Piastri establishes the main advantage of the final sector. Norris enters faster at 121.9 km/h compared to Piastri's 111.0 km/h, but drops to a lower minimum speed of 110.1 km/h. Piastri maintains 111.2 km/h through the minimum and commits more strongly to the throttle, recording 78.5% mean throttle and 56.3% full throttle compared to Norris's 74.5% and 50.0%. The difference is reflected in the exit, where Piastri reaches 156.3 km/h against Norris's 144.2 km/h.

Piastri then carries this momentum into Turn 18, entering at 188.1 km/h compared to Norris's 176.8 km/h. He maintains a higher speed through the corner, reaching a minimum of 192.4 km/h and exiting at 238.4 km/h, compared to Norris's 183.4 km/h minimum and 233.5 km/h exit. This adds a further advantage and allows Piastri to finish the final sequence strongly despite Norris's gains in Turns 15 and 16.


## Case Study Conclusion

The aggregate telemetry analysis shows that the performance difference between Piastri and Norris is concentrated in specific corner sequences rather than being driven by one consistent advantage across the lap. Norris gains time through several high-speed sections by carrying more speed into the corner, while Piastri recovers time through stronger exits and more effective throttle application in key braking and acceleration zones.

Overall, the telemetry shows a highly variable comparison between the two drivers. Norris tends to gain through higher entry and minimum speeds in several corners, while Piastri produces some of his largest gains through stronger exits and better acceleration out of key sections. The lap is therefore shaped by a series of opposing gains and losses, with the largest differences concentrated around specific corners rather than following one uniform driving pattern.


# Conclusion

The combined findings from both case studies show that Formula 1 lap time is shaped by the interaction between corner entry, braking, apex speed, and exit performance rather than by any single driving metric. Across both intra-team comparisons, the telemetry reveals that gaining speed at one stage of a corner does not necessarily translate into a corresponding lap time advantage. Performance depends on how effectively the different phases of the corner are connected.

The analysis also highlights the trade-offs involved in different driving approaches. Higher entry speeds can provide an immediate advantage, but may require longer or more aggressive deceleration phases. Conversely, sacrificing entry speed can allow a driver to manage the car differently through the apex and potentially recover speed on exit. These effects vary from corner to corner, meaning that a faster entry, apex, or exit speed should not be evaluated independently from the rest of the corner.

The scatter and delta analyses further demonstrate that the largest performance differences are concentrated in specific sections rather than being distributed uniformly across a lap. This reinforces the importance of analysing the complete corner sequence, from braking and entry through apex and throttle application to the final exit speed.

Ultimately, the telemetry shows that driver performance cannot be judged by braking points or entry speeds alone. The most meaningful indicator of performance is how effectively a driver balances the different phases of each corner to produce a competitive overall lap time. In modern Formula 1, extracting performance is therefore less about maximizing a single metric and more about managing the compromises between entry speed, car rotation, apex speed, throttle application, and exit performance.